# Predicting Air Turbulence Using Machine Learning

By: Chee Qian Wen
<br>Feb 2025
## Introduction
Air turbulence refers to the irregular and often unpredictable movement of air masses within the Earth's atmosphere. These turbulent air currents can cause aircrafts to experience sudden jolts, creating a bumpy ride for passengers.

Due to climate change, incidences of air turbulence have been on the rise in recent years. Understanding and being able to predict air turbulence is crucial for several reasons:
* Passenger Safety and Comfort: Turbulence, while usually not dangerous, can cause injuries if passengers are not securely fastened. Predicting turbulence allows pilots to warn passengers and crew to secure their seatbelts, minimizing the risk of injury.

* Flight Efficiency: By predicting areas of turbulence, pilots can plan more efficient flight routes, leading to smoother flights and potentially saving fuel. Avoiding turbulence can also reduce wear and tear on the aircraft.

* Operational Planning: Airlines can use turbulence predictions to make informed decisions about flight schedules, routing, and fuel requirements, optimizing operational efficiency and reducing delays.

The purpose of this analysis is therefore to look at the features influencing air turbulence, and to build a model that can accurately predict air turbulence based on these features.

This analysis focuses on turbulence over the past 10 years (Jan 2015 - Dec 2024) over US mainland only.

There are 5 notebooks to this analyses:
1) Part 1: Importing and Extracting Data
2) Part 2: Data Cleaning and Merging
3) Part 3: Feature Engineering
4) Part 4: Exploratory Data Analyses (EDA)
5) Part 5: Modelling

## Part 1 of 5: Importing and Extracting Data
Part 1 involves extracting the relevant data required for this analysis.

### 1.1. Import Libraries
First, the necessary libraries required are imported.

In [ ]:
import requests
import pandas as pd
import glob
import datetime

### 1.2. Import Pilot Reports (PIREPs)

PIREPs, or Pilot Reports, are real-time weather reports filed by pilots during their flights. These reports provide valuable information about actual weather conditions encountered in the air, such as turbulence, icing, visibility, and wind shear. PIREPs are crucial for enhancing flight safety as they help other pilots and air traffic control (ATC) make informed decisions to avoid hazardous weather conditions.

In this section, data is extracted from Iowa State University's archive of PIREPs (https://mesonet.agron.iastate.edu/request/gis/pireps.php) over US mainland using Application Programming Interface (API). Search is limit to the past 10 years (2015-2024).

This data contains the following information:
* Date and time in UTC
* Coordinates of the reported incident (longitude, latitude)
* Flight level
* Aircraft type
* Turbulence report (description of turbulence)

In [ ]:
# specify base url
url = "https://mesonet.agron.iastate.edu/cgi-bin/request/gis/pireps.py"

# specify list of start and end dates with corresponding filenames
date_ranges = [
    ("2015-01-01T00:00:00Z", "2015-07-01T00:00:00Z", "pireps2015-1.csv"),
    ("2015-07-01T00:00:00Z", "2016-01-01T00:00:00Z", "pireps2015-2.csv"),
    ("2016-01-01T00:00:00Z", "2016-07-01T00:00:00Z", "pireps2016-1.csv"),
    ("2016-07-01T00:00:00Z", "2017-01-01T00:00:00Z", "pireps2016-2.csv"),
    ("2017-01-01T00:00:00Z", "2017-07-01T00:00:00Z", "pireps2017-1.csv"),
    ("2017-07-01T00:00:00Z", "2018-01-01T00:00:00Z", "pireps2017-2.csv"),
    ("2018-01-01T00:00:00Z", "2018-07-01T00:00:00Z", "pireps2018-1.csv"),
    ("2018-07-01T00:00:00Z", "2019-01-01T00:00:00Z", "pireps2018-2.csv"),
    ("2019-01-01T00:00:00Z", "2019-07-01T00:00:00Z", "pireps2019-1.csv"),
    ("2019-07-01T00:00:00Z", "2020-01-01T00:00:00Z", "pireps2019-2.csv"),
    ("2020-01-01T00:00:00Z", "2020-07-01T00:00:00Z", "pireps2020-1.csv"),
    ("2020-07-01T00:00:00Z", "2021-01-01T00:00:00Z", "pireps2020-2.csv"),
    ("2021-01-01T00:00:00Z", "2021-07-01T00:00:00Z", "pireps2021-1.csv"),
    ("2021-07-01T00:00:00Z", "2022-01-01T00:00:00Z", "pireps2021-2.csv"),
    ("2022-01-01T00:00:00Z", "2022-07-01T00:00:00Z", "pireps2022-1.csv"),
    ("2022-07-01T00:00:00Z", "2023-01-01T00:00:00Z", "pireps2022-2.csv"),
    ("2023-01-01T00:00:00Z", "2023-07-01T00:00:00Z", "pireps2023-1.csv"),
    ("2023-07-01T00:00:00Z", "2024-01-01T00:00:00Z", "pireps2023-2.csv"),
    ("2024-01-01T00:00:00Z", "2024-07-01T00:00:00Z", "pireps2024-1.csv"),
    ("2024-07-01T00:00:00Z", "2025-01-01T00:00:00Z", "pireps2024-2.csv"),
]

# loop through dates and save data as a separate csv file for each loop
for sts, ets, filename in date_ranges:
    params = {
        "sts": sts,
        "ets": ets,
        "artcc": "_ALL",
        "fmt": "csv"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
    else:
        print(f"Request failed for date range {sts} to {ets}: {response.status_code}")

In [ ]:
# after all csv files have been obtained, specify which data to keep
csv_files = ["pireps2015-1.csv",
             "pireps2015-2.csv",
             "pireps2016-1.csv",
             "pireps2016-2.csv",
             "pireps2017-1.csv",
             "pireps2017-2.csv",
             "pireps2018-1.csv",
             "pireps2018-2.csv",
             "pireps2019-1.csv",
             "pireps2019-2.csv",
             "pireps2020-1.csv",
             "pireps2020-2.csv",
             "pireps2021-1.csv",
             "pireps2021-2.csv",
             "pireps2022-1.csv",
             "pireps2022-2.csv",
             "pireps2023-1.csv",
             "pireps2023-2.csv",
             "pireps2024-1.csv",
             "pireps2024-2.csv"
            ]

# list to store DataFrames
df_list = []

# specify columns to exclude
exclude_columns = ['REPORT','ICING','PRODUCT_ID','ATRCC']

# read each CSV file, drop the specified columns, and select only rows with turbulence data
for file in csv_files:
    df = pd.read_csv(file, low_memory = False)
    df = df.drop(columns=exclude_columns)
    df = df[(df['TURBULENCE'].notna())]
    df['TURBULENCE'] = df['TURBULENCE'].astype(str)
    df = df[(~df['TURBULENCE'].str.contains('NEG', regex=True)) & 
         (~df['TURBULENCE'].str.contains('SMOOTH', regex=True)) &
         (~df['TURBULENCE'].str.contains('SMTH', regex=True))]
    df_list.append(df)

# concatenate all dataFrames into one
df_all = pd.concat(df_list, ignore_index=True)
df_all = df_all.drop_duplicates()

In [ ]:
# save dataframe as a csv file
df_all.to_csv('PIREPs_all.csv', index = False)

### 1.3. Import Elevation Data
Considering ground elevation helps pilots and meteorologists predict areas where turbulence is likely to occur. Ground elevation plays a significant role in predicting turbulence for several reasons:

* Orographic Turbulence: When strong winds flow over mountainous or hilly terrain, the terrain can disturb the airflow, creating turbulence. This type of turbulence, known as orographic turbulence, is common near mountain ranges.

* Atmospheric Stability: The stability of the atmosphere can be affected by ground elevation. For example, higher elevations can lead to cooler temperatures and different atmospheric conditions, influencing the likelihood and intensity of turbulence.

* Wind Patterns: Elevation changes can alter local wind patterns. Wind flowing over elevated terrain can create complex wind patterns, including updrafts and downdrafts, which contribute to turbulence.

* Thermal Turbulence: During the day, the sun heats the ground, causing warm air to rise. This rising warm air can create thermal turbulence. The effect can be more pronounced over elevated terrain, where the ground heats up faster and more unevenly.

In this section, elevation data is extracted from the United States Geological Survey (USGS) website (https://apps.nationalmap.gov/epqs/) using API. This API returns the elevation (in meters) for a specified coordinate (longitude, latitude).

NOTE: This API has a throttle and runs very slowly. 

In [ ]:
# first save the coordinates from the PIREPs as a new csv file
dfpireps = pd.read_csv('PIREPs_all.csv', low_memory = False)

# extract LON and LAT columns
df_lonlat = df[['LON', 'LAT']]

# round values to 1 decimal place to reduce processing time
df_lonlat = df_lonlat.round(1)

# save coordinates as a csv file
df_lonlat.to_csv('lonlat.csv', index=False)

In [ ]:
df_lonlat = pd.read_csv('lonlat.csv')

# List to store elevation data
elevations = []

# iterate through coordinate in the dataframe
for index, row in df.iterrows():
    lon = row['LON']
    lat = row['LAT']
    url = 'https://epqs.nationalmap.gov/v1/json'
    params = {
        "x": lon,
        "y": lat,
        "wkid": '4326',
        "units": 'Meters',
        "includeDate": 'false'
    }
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        elevation = response.json().get('value', None)
    except (requests.exceptions.RequestException, ValueError):
        print(f"Error for LON: {lon}, LAT: {lat}")
        elevation = 'NA'
    elevations.append(elevation)

# add the elevation data to the DataFrame
df_lonlat['Elevation'] = elevations

# save elevation data as a csv file
df_lonlat.to_csv('elevation.csv')

### 1.4. Extract Aircraft Specifications
Aircraft specifications such as aircraft type, weight class, engine type, and the number of engines can be critical factors in predicting turbulence for several reasons:

* Aircraft type: Different types of aircraft have varying aerodynamic designs, structures, and capabilities. For instance, smaller aircraft may be more susceptible to turbulence compared to larger, sturdier commercial jets. Understanding the type of aircraft helps in assessing how it will respond to turbulent conditions.

* Weight class: The weight of an aircraft significantly influences its stability during flight. Heavier aircraft tend to be more stable and less affected by turbulence, while lighter aircraft might experience more pronounced effects. Knowing the weight class helps predict the severity of turbulence an aircraft might encounter.

* Engine type: The type of engines installed on an aircraft can impact its performance and handling during turbulent conditions. For example, turbojet engines may respond differently to turbulence compared to turboprop engines. Engine characteristics, such as thrust and responsiveness, can play a role in how the aircraft maneuvers through turbulent air.

* Number of engines: The number of engines on an aircraft can affect its overall power and redundancy. Aircraft with multiple engines might have better capabilities to handle turbulence and maintain stability compared to single-engine aircraft. Additionally, the distribution of engines on the wings or fuselage can influence the aircraft's aerodynamic stability.

Including aircraft specifications can therefore enhance the model's ability to predict turbulence.

Proceed to the International Civil Aviation Organization (ICAO) website (https://www.icao.int/publications/DOC8643/Pages/Search.aspx) to download aircraft specs for each aircraft type.

### 1.5. Import Meteoreological Data

Meteorological data, including wind speed, wind direction, temperature, and humidity, play a significant role in affecting turbulence:
* Wind speed: Variations in wind speed, especially sudden changes, can create turbulent conditions. When aircraft move through areas with fluctuating wind speeds, it can cause the plane to experience jolts and shakes.

* Wind direction: Changes in wind direction can lead to turbulence, particularly when flying across weather fronts or near mountains. Wind direction shifts can cause unstable airflows, resulting in turbulence.

* Temperature: Temperature gradients, such as those found between warm and cold air masses, can lead to turbulence. For example, when an aircraft flies from a warm area to a cooler one, the differing air densities can cause unstable air currents.

* Humidity: Moisture levels in the air can also impact turbulence. High humidity levels can contribute to the formation of clouds and storms, which are often associated with turbulent conditions. Additionally, the presence of water vapor can influence air density and stability.

In this section, daily weather is extracted from Iowa State University's archive of the U.S. National Weather Service data (https://mesonet.agron.iastate.edu/request/daily.phtml#) from thousands of stations across the country. Data is extracted using API. This API returns the daily maximum and minimum temperature, average wind speed, average wind direction, relative humidity and precipitation (in inches) for each station.

In [ ]:
# specify base url
url = "https://mesonet.agron.iastate.edu/cgi-bin/request/daily.py"

# import list of networks in the US
networks = pd.read_csv('network.csv')

# define date range
start_date = datetime.datetime(2015, 1, 1)
end_date = datetime.datetime(2024, 12, 31)

for network in networks['Network']:
    all_data = []

    # loop through each year from 2015 to 2024
    current_date = start_date
    while current_date < end_date:
        next_year = current_date + datetime.timedelta(days=365)
        if next_year > end_date:
            next_year = end_date
        
        params = {
            "sts": current_date.strftime('%Y-%m-%d'),
            "ets": next_year.strftime('%Y-%m-%d'),
            "network": network,
            "stations": "_ALL",
            "var": "max_temp_f,min_temp_f,avg_wind_speed_kts,avg_wind_drct,avg_rh,precip_in",
            "format" : "csv"
        }

        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            all_data.append(response.content)
        else:
            print(f"Request failed for network {network}: {response.status_code}")
        
        current_date = next_year

    filename = f"{network}_weather.csv"
    
    with open(filename, "wb") as f:
        for data in all_data:
            f.write(data)

In [ ]:
# first process ASOS weather
asos_csv_files = glob.glob("*ASOS_weather.csv")

additional_files = ['VTWAC_weather.csv','USCRN_weather.csv']

all_files = asos_csv_files + additional_files
                    
# import each csv file as a df
weather_dfs = [pd.read_csv(file, low_memory = False) for file in all_files]

# combine all dataframes into a single dataframe
weather_df = pd.concat(weather_dfs, ignore_index=True)

In [ ]:
# drop all rows with no data
columns_to_check = ['max_temp_f', 'min_temp_f', 'precip_in', 'avg_wind_speed_kts', 'avg_wind_drct', 'avg_rh']

for col in columns_to_check:
    weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')
    
weather_df.dropna(subset=columns_to_check, how ='all', inplace=True)

Next, the location of each weather station must be downloaded to know which station is closest to each pilot report. Latitude and longitude coordinates of the weather stations can be downloaded from https://mesonet.agron.iastate.edu/sites/networks.php?network=_ALL_&format=csv.

In [ ]:
# import station locations as df
stations = pd.read_csv('station_latlon.csv')

df = pd.merge(weather_df, stations, on ='station', how ='left')

df = df.dropna(subset=['st_lon', 'st_lat'])
df = df[(df['st_lon'] != 0) & (df['st_lat'] != 0)]

# replace null precipitation as 0
df['precip_in'] = df['precip_in'].fillna(0)

df = df.drop_duplicates(subset=['day', 'st_lon', 'st_lat'])

In [ ]:
# save all weather information with locations to a new CSV file
df.to_csv("weather_all.csv", index=False)

## Proceed to Part 2.

There are now 4 raw datasets:
1) 'PIREPs_all.csv'
2) 'elevation.csv'
3) 'ICAO_aircraft_type.csv'
4) 'weather_all.csv'

Part 2 explains how the data are cleaned and merged into a single dataset.